In [ ]:
from fastai.vision.all import *
from torch.nn import *
from itertools import permutations

set_num_threads(1)
torch.set_num_threads(1)


class F(Module):
    def __init__(s, f, *a):
        super().__init__()
        s.f = f
        s.a = ParameterList([Parameter(a) for a in a])

    def forward(s, x):
        return s.f(x, *s.a)


class Red(Sequential):
    def forward(s, x):
        while x.size(2) > 1:
            if x.size(2) % 2:
                x, y = x.split([x.size(2) - 1, 1], 2)
            else:
                y = None
            x = super().forward(x.reshape(len(x), -1, x.size(3) * 2)).reshape(*x.shape[:2], -1, x.size(3))
            if y is not None:

                x = torch.cat((x, y), 2)
        return x[:, :, 0]


class Red1(Sequential):
    def forward(s, x):
        y = super().forward(x[:, :, :2].view(*x.shape[:2], -1))
        for x in x[:, :, 2:].permute(2, 0, 1, 3):
            y = super().forward(torch.cat((y, x), 2))
        return y


def MGLinear(m, g, k, K):
    return F(
        lambda x, w, i=torch.tensor(list(permutations(range(g)))): (x @ w[:, :, i].reshape(m, -1, g * k).transpose(1, 2)).reshape(*x.shape[:2], K, -1),
        init.kaiming_uniform_(torch.empty(m, K, g, k), 5 ** 0.5),
    )


def MBatchNorm(m, k):
    return F(
        lambda x, w, b: torch.batch_norm(x.transpose(1, 2).reshape(1, m * k, -1), w, b, None, None, True, 0, 0, True)
        .reshape(m, k, -1, x.size(3))
        .transpose(1, 2),
        torch.ones(m * k),
        torch.zeros(m * k),
    )


def MLinear(m, k, K):
    return F(lambda x, w, b: b.baddbmm(x, w.transpose(1, 2)), init.kaiming_uniform_(torch.empty(m, K, k), 5 ** 0.5), torch.zeros(m, 1, K))


class dat:
    def __init__(s, e, l, n, it, dev, **_):
        s.e = e
        s.l = l
        s.n = n
        s.it = it
        s.dev = dev

    def __len__(s):
        return s.it

    def __iter__(s):
        for _ in range(s.it):
            _ = torch.randint(s.e, [s.n, s.l], device=s.dev)
            yield _, _.sum(1) & (s.e - 1)


class Los(NLLLoss):
    def forward(s, y, t):
        return super().forward(y.log_softmax(2).mean(0), t)


def acc(n):
    def avg(y, t):
        nonlocal _
        t = (y.argmax(2) == t).float().mean(1)
        _ = torch.zeros(n, dtype=torch.long, device=y.device).put_(
            *torch.broadcast_tensors((t * (n - 1e-3)).long(), torch.tensor(1, device=y.device)), True
        ).float() * (100 / len(y))
        return t.mean() * 100

    yield avg
    for i, j in enumerate(np.linspace(0, 100, n, 0, dtype=int)):

        def _(y, t, i=i):
            return _[i]

        _.__name__ = str(j)
        yield _


def learn(m, e, k, l, n=16, dev="cuda"):
    return Learner(
        DataLoaders(dat(e, l, n, 9999, dev), dat(e, l, 1024, 1, dev)),
        Sequential(
            F(lambda x, w: w[:, x], Parameter(torch.randn(m, e, k))),
            Red1(
                MGLinear(m, 2, k, k),
                MBatchNorm(m, k),
                Swish(),
                F(lambda x: x.sum(3)),
            ),
            MLinear(m, k, e),
        ).to(dev),
        Los(),
        metrics=acc(10),
        cbs=ShowGraphCallback,
    )

In [ ]:
m = learn(1024, 16, 4, 4, dev='cuda')
m.fit_one_cycle(20, 0.001, 9, 99, 0.01)

In [ ]:
m.fit_one_cycle(200, 0.001, 9, 99, 0.01)